In [1]:
"""
UTS Machine learning
WPM C Kelompok 3

 Ketua Kelompok   : 
     PETER KINZA PRISDIANTO (230935604944)
 Anggota Kelompok :
	SATRIA FERDI ANANDA (230532601181)
	YUSUF BAH'TIAR (230935603592)

"""

"\nUTS Machine learning\nWPM C Kelompok 3\n\n Ketua Kelompok   : \n     PETER KINZA PRISDIANTO (230935604944)\n Anggota Kelompok :\n\tSATRIA FERDI ANANDA (230532601181)\n\tYUSUF BAH'TIAR (230935603592)\n\n"

In [2]:
# Menampilkan folder tempat JupyterLab menjalankan kode Python.
import os

print("Direktori kerja saat ini:")
print(os.getcwd())

# Menampilkan daftar file dan folder di direktori kerja.
print("\nIsi direktori kerja:")
print(os.listdir())

""" +++"""

# Mencari file Dataset_DHT22.xlsx di direktori kerja
# dan seluruh subfolder di bawahnya.
import os

nama_file = "Dataset_DHT22.xlsx"
hasil_pencarian = []

for folder, subfolder, daftar_file in os.walk(os.getcwd()):
    if nama_file in daftar_file:
        hasil_pencarian.append(
            os.path.join(folder, nama_file)
        )

# Menampilkan hasil pencarian.
if hasil_pencarian:
    print("File ditemukan:")
    for path in hasil_pencarian:
        print(path)
else:
    print("File tidak ditemukan di direktori kerja dan subfoldernya.")


Direktori kerja saat ini:
C:\Users\Samsung\folder dataset uts

Isi direktori kerja:
['.ipynb_checkpoints', 'Dataset_DHT22.xlsx', 'UTS WPM K-3 1.1.ipynb', 'UTS WPM K-3.ipynb']
File ditemukan:
C:\Users\Samsung\folder dataset uts\Dataset_DHT22.xlsx


In [4]:
%pip install -U ipywidgets jupyterlab_widgets

Note: you may need to restart the kernel to use updated packages.


In [3]:

# ============================================================
# CELL 1: PERSIAPAN DAN IMPORT LIBRARY
# ============================================================

# Memasang library yang diperlukan.
# %pip digunakan di JupyterLab untuk memasang library
# pada environment Python yang digunakan oleh notebook.
%pip install openpyxl pandas numpy scikit-learn ipywidgets

# ------------------------------------------------------------
# 1. Import library untuk mengolah data
# ------------------------------------------------------------

import os  # Memeriksa keberadaan file pada folder kerja.

import pandas as pd  # Membaca, membersihkan, dan mengolah data tabel.
import numpy as np  # Membantu operasi numerik dan pemeriksaan nilai.

# ------------------------------------------------------------
# 2. Import library untuk membagi data dan membuat model KNN
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split
# Membagi data menjadi data latih dan data uji.

from sklearn.preprocessing import StandardScaler
# Menyamakan skala fitur suhu dan kelembapan.

from sklearn.neighbors import KNeighborsClassifier
# Menggunakan algoritma K-Nearest Neighbors untuk klasifikasi.

from sklearn.pipeline import Pipeline
# Menggabungkan proses scaling dan model KNN dalam satu alur.

# ------------------------------------------------------------
# 3. Import library untuk evaluasi model
# ------------------------------------------------------------

from sklearn.metrics import accuracy_score, classification_report
# accuracy_score menghitung akurasi model.
# classification_report menampilkan precision, recall, dan F1-score.

# ------------------------------------------------------------
# 4. Import library untuk antarmuka prediksi
# ------------------------------------------------------------

import ipywidgets as widgets
# Membuat komponen antarmuka, seperti kolom input dan tombol.

from IPython.display import display, clear_output
# display menampilkan antarmuka di notebook.
# clear_output membersihkan keluaran sebelumnya.

# ------------------------------------------------------------
# 5. Memeriksa folder kerja notebook
# ------------------------------------------------------------

print("Folder kerja saat ini:")
print(os.getcwd())

print("\nPersiapan library selesai.")


Note: you may need to restart the kernel to use updated packages.
Folder kerja saat ini:
C:\Users\Samsung\folder dataset uts

Persiapan library selesai.


In [5]:

# ============================================================
# CELL 2: FULL PIPELINE MACHINE LEARNING MENGGUNAKAN KNN
# ============================================================
# Pipeline ini mencakup:
# 1. Instalasi dan import library
# 2. Membaca dataset Excel
# 3. Menggabungkan seluruh sheet
# 4. Membersihkan data sensor
# 5. Membuat label Panas dan Dingin
# 6. Memisahkan data training dan testing
# 7. Melatih serta mengevaluasi model KNN
# 8. Melatih model final untuk digunakan pada Cell 3
#
# ATURAN LABEL:
# Suhu < 20°C  = Dingin
# Suhu > 25°C  = Panas
# Suhu 20–25°C = belum memiliki label awal
#
# Catatan:
# Model belajar dari label berdasarkan aturan suhu di atas.
# Prediksi pada rentang 20–25°C merupakan perkiraan KNN,
# bukan label yang sudah terbukti benar.
# ============================================================


# BAGIAN 1: MENYIAPKAN LIBRARY
# %pip memastikan library yang diperlukan tersedia di environment
# Python yang digunakan oleh notebook ini.
%pip install -q openpyxl scikit-learn pandas numpy ipywidgets

import os                         # Memeriksa lokasi dan keberadaan file.
import math                       # Membantu menentukan ukuran data testing.
import numpy as np                # Operasi numerik dan pemeriksaan nilai valid.
import pandas as pd               # Membaca serta mengolah dataset berbentuk tabel.

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# BAGIAN 2: MENENTUKAN FILE DATASET
# Nama file digunakan karena dataset sebelumnya ditemukan
# dalam direktori kerja notebook JupyterLab.
file_path = "Dataset_DHT22.xlsx"

# Memeriksa keberadaan file sebelum membacanya.
# Dengan pemeriksaan ini, kesalahan lokasi file diketahui sejak awal.
if not os.path.isfile(file_path):
    raise FileNotFoundError(
        f"File '{file_path}' tidak ditemukan.\n"
        f"Direktori kerja saat ini: {os.getcwd()}\n"
        "Letakkan file Excel di folder notebook atau perbaiki file_path."
    )

print("Dataset ditemukan:", os.path.abspath(file_path))


# BAGIAN 3: MEMBACA DAN MENGGABUNGKAN SELURUH SHEET EXCEL
# ExcelFile digunakan untuk mengetahui sheet yang tersedia.
# Seluruh sheet dibaca agar pengukuran dari setiap sesi dapat digunakan.
excel_file = pd.ExcelFile(file_path)

daftar_data = []

for nama_sheet in excel_file.sheet_names:
    # Membaca data dari satu sheet.
    data_sheet = pd.read_excel(
        file_path,
        sheet_name=nama_sheet
    )

    # Merapikan nama kolom untuk menghindari spasi yang tidak perlu.
    data_sheet.columns = data_sheet.columns.astype(str).str.strip()

    # Menyimpan nama sheet sebagai informasi asal pengukuran.
    # Informasi ini tidak digunakan sebagai fitur KNN.
    data_sheet["Sesi"] = nama_sheet

    daftar_data.append(data_sheet)

# Menggabungkan semua sheet menjadi satu dataset.
# ignore_index=True membuat nomor baris tersusun kembali.
df = pd.concat(daftar_data, ignore_index=True)

print("\nDaftar sheet:", excel_file.sheet_names)
print("Ukuran dataset gabungan:", df.shape)
print("\nContoh data:")
display(df.head())


# BAGIAN 4: MEMERIKSA KOLOM DAN MEMBERSIHKAN DATA
# KNN memerlukan fitur numerik yang valid untuk menghitung jarak.
# Karena itu, kolom suhu dan kelembapan harus tersedia dan numerik.
fitur = ["Suhu (°C)", "Kelembapan (%)"]

for kolom in fitur:
    if kolom not in df.columns:
        raise ValueError(
            f"Kolom '{kolom}' tidak ditemukan.\n"
            f"Kolom yang tersedia: {df.columns.tolist()}"
        )

# Mengubah fitur sensor menjadi angka.
# Isi yang tidak bisa dikonversi akan menjadi NaN.
for kolom in fitur:
    df[kolom] = pd.to_numeric(
        df[kolom],
        errors="coerce"
    )

# Menghapus baris yang suhu atau kelembapannya kosong.
# KNN tidak dapat menghitung jarak dari fitur yang bernilai NaN.
jumlah_awal = len(df)

df = df.dropna(subset=fitur).copy()

# Menghapus nilai tak hingga karena bukan nilai sensor numerik yang valid.
mask_valid = np.isfinite(df[fitur].to_numpy(dtype=float)).all(axis=1)
df = df.loc[mask_valid].copy()

print("\nJumlah baris sebelum pembersihan:", jumlah_awal)
print("Jumlah baris setelah pembersihan:", len(df))
print("Jumlah baris yang dikeluarkan:", jumlah_awal - len(df))
print("\nStatistik fitur sensor:")
display(df[fitur].describe())

if len(df) < 2:
    raise ValueError("Data valid terlalu sedikit untuk diproses.")


# BAGIAN 5: MEMBUAT LABEL PANAS DAN DINGIN
# Dataset awal belum mempunyai kolom target.
# Kita membuat target berdasarkan aturan yang telah ditentukan.
# Data pada rentang 20–25°C dibiarkan tanpa label.
df["Kondisi"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)

# Memberi label Dingin untuk suhu di bawah 20°C.
df.loc[df["Suhu (°C)"] < 20, "Kondisi"] = "Dingin"

# Memberi label Panas untuk suhu di atas 25°C.
df.loc[df["Suhu (°C)"] > 25, "Kondisi"] = "Panas"

# Data tanpa label tidak digunakan untuk training maupun evaluasi.
df_berlabel = df.dropna(subset=["Kondisi"]).copy()

# Menghitung jumlah data untuk setiap kelas.
print("\nJumlah data berlabel:")
print(df_berlabel["Kondisi"].value_counts())

print("\nJumlah data yang belum berlabel (20–25°C):",
      df["Kondisi"].isna().sum())

# Pastikan kelas Panas dan Dingin sama-sama tersedia.
if df_berlabel["Kondisi"].nunique() != 2:
    raise ValueError(
        "Training membutuhkan data berlabel untuk kelas Panas dan Dingin.\n"
        "Periksa apakah dataset memiliki suhu di bawah 20°C "
        "dan suhu di atas 25°C."
    )

# Setiap kelas perlu memiliki setidaknya dua contoh agar data
# dapat dibagi untuk training dan testing.
jumlah_per_kelas = df_berlabel["Kondisi"].value_counts()

if jumlah_per_kelas.min() < 2:
    raise ValueError(
        "Setiap kelas harus memiliki minimal dua data berlabel "
        "untuk pembagian training dan testing."
    )


# BAGIAN 6: MEMISAHKAN DATA TRAINING DAN TESTING
# X adalah fitur yang dibaca model, yaitu suhu dan kelembapan.
# y adalah target yang dipelajari model, yaitu Panas atau Dingin.
X = df_berlabel[fitur]
y = df_berlabel["Kondisi"]

# Pembagian data testing dibuat minimal dua baris agar kedua kelas
# dapat terwakili saat evaluasi, selama jumlah data memungkinkan.
# random_state=42 membuat pembagian dapat diulang.
# stratify=y mempertahankan proporsi kelas sebisa mungkin.
ukuran_test = max(2, math.ceil(0.20 * len(y)))
ukuran_test = min(ukuran_test, len(y) - 2)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=ukuran_test,
    random_state=42,
    stratify=y
)

print("\nJumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))


# BAGIAN 7: MEMBUAT, MELATIH, DAN MENGEVALUASI MODEL KNN
# KNN menggunakan jarak untuk menemukan tetangga terdekat.
# StandardScaler menyamakan skala suhu dan kelembapan supaya
# fitur dengan rentang angka lebih besar tidak mendominasi jarak.
#
# Pipeline memastikan scaling pada training dan inference konsisten.
# k=5 adalah baseline awal, bukan hasil optimasi nilai k.
k = min(5, len(X_train))

model_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=k))
])

# Melatih pipeline pada data training.
model_knn.fit(X_train, y_train)

# Menguji model menggunakan data testing yang tidak dipakai saat fit.
y_pred = model_knn.predict(X_test)

# Menghitung akurasi untuk mengukur proporsi prediksi yang cocok
# dengan label aturan pada data testing.
akurasi = accuracy_score(y_test, y_pred)

print("\n========== EVALUASI MODEL BASELINE ==========")
print("Algoritma: K-Nearest Neighbors (KNN)")
print("Jumlah tetangga (k):", k)
print(f"Akurasi baseline: {akurasi:.4f} ({akurasi * 100:.2f}%)")

print("\nLaporan klasifikasi:")
print(classification_report(
    y_test,
    y_pred,
    labels=["Dingin", "Panas"],
    zero_division=0
))

print("Confusion matrix (urutan kelas: Dingin, Panas):")
print(confusion_matrix(
    y_test,
    y_pred,
    labels=["Dingin", "Panas"]
))


# BAGIAN 8: MELATIH MODEL FINAL UNTUK INFERENCE
# Setelah evaluasi baseline selesai, model final dilatih kembali
# menggunakan seluruh data yang sudah memiliki label.
# Tujuannya agar inference menggunakan semua contoh berlabel.
# Evaluasi di atas tetap berasal dari model sebelum pelatihan ulang.
model_knn.fit(X, y)

print("\nModel final berhasil dilatih menggunakan seluruh data berlabel.")
print("Variabel model siap digunakan pada Cell 3.")
print("Fitur input:", fitur)
print("Kelas model:", model_knn.named_steps["knn"].classes_)


Note: you may need to restart the kernel to use updated packages.
Dataset ditemukan: C:\Users\Samsung\folder dataset uts\Dataset_DHT22.xlsx

Daftar sheet: ['Dataset ML DHT22', 'Dataset ML DHT22 (2)']
Ukuran dataset gabungan: (200, 4)

Contoh data:


,Waktu,Suhu (°C),Kelembapan (%),Sesi
0,06:00:00,22.7,82.4,Dataset ML DHT22
1,06:15:00,23.0,81.6,Dataset ML DHT22
2,06:30:00,23.4,78.8,Dataset ML DHT22
3,06:45:00,24.0,81.2,Dataset ML DHT22
4,07:00:00,24.8,78.2,Dataset ML DHT22



Jumlah baris sebelum pembersihan: 200
Jumlah baris setelah pembersihan: 200
Jumlah baris yang dikeluarkan: 0

Statistik fitur sensor:


,Suhu (°C),Kelembapan (%)
count,200.000000,200.000000
mean,26.383000,73.862500
std,4.341191,11.693607
min,19.500000,53.300000
25%,22.400000,63.125000
50%,26.100000,74.200000
75%,30.525000,85.000000
max,33.900000,93.000000



Jumlah data berlabel:
Kondisi
Panas     110
Dingin      2
Name: count, dtype: Int64

Jumlah data yang belum berlabel (20–25°C): 88

Jumlah data training: 89
Jumlah data testing: 23

========== EVALUASI MODEL BASELINE ==========
Algoritma: K-Nearest Neighbors (KNN)
Jumlah tetangga (k): 5
Akurasi baseline: 1.0000 (100.00%)

Laporan klasifikasi:
              precision    recall  f1-score   support

      Dingin       0.00      0.00      0.00         0
       Panas       1.00      1.00      1.00        23

    accuracy                           1.00        23
   macro avg       0.50      0.50      0.50        23
weighted avg       1.00      1.00      1.00        23

Confusion matrix (urutan kelas: Dingin, Panas):
[[ 0  0]
 [ 0 23]]

Model final berhasil dilatih menggunakan seluruh data berlabel.
Variabel model siap digunakan pada Cell 3.
Fitur input: ['Suhu (°C)', 'Kelembapan (%)']
Kelas model: ['Dingin' 'Panas']


In [6]:

# ============================================================
# CELL 3: ANTARMUKA INPUT DAN PREDIKSI KNN
# ============================================================
# Antarmuka menyediakan:
# 1. Kolom input suhu
# 2. Kolom input kelembapan
# 3. Tombol Proses
# 4. Area hasil klasifikasi
#
# Saat tombol ditekan, nilai input dikirim ke model_knn
# dari Cell 2 untuk diprediksi menjadi Panas atau Dingin.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# Memastikan Cell 2 sudah berhasil membuat model.
# Pemeriksaan ini mencegah antarmuka digunakan sebelum training.
if "model_knn" not in globals():
    raise RuntimeError(
        "Model belum tersedia. Jalankan Cell 2 terlebih dahulu."
    )

# Membuat kolom input suhu.
# FloatText memungkinkan pengguna mengetik nilai suhu desimal.
input_suhu = widgets.FloatText(
    value=25.0,
    description="Suhu (°C):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

# Membuat kolom input kelembapan.
# Nilai awal 60 hanya contoh input dan dapat diganti pengguna.
input_kelembapan = widgets.FloatText(
    value=60.0,
    description="Kelembapan (%):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

# Membuat tombol yang akan menjalankan proses inference.
tombol_proses = widgets.Button(
    description="Proses",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="160px")
)

# Area untuk menampilkan hasil tanpa mengubah tampilan input.
area_hasil = widgets.Output()

# Fungsi ini dijalankan setiap kali tombol Proses ditekan.
def proses_prediksi(tombol):
    with area_hasil:
        # Membersihkan hasil prediksi sebelumnya agar hasil baru
        # tidak menumpuk pada setiap klik.
        clear_output(wait=True)

        # Membaca nilai dari kedua kolom input.
        suhu = input_suhu.value
        kelembapan = input_kelembapan.value

        # Memastikan input merupakan nilai numerik yang valid.
        if not np.isfinite(suhu) or not np.isfinite(kelembapan):
            print("Error: masukkan angka suhu dan kelembapan yang valid.")
            return

        # Memeriksa rentang kelembapan yang lazim digunakan
        # dalam persentase. Nilai di luar 0–100 ditolak.
        if not 0 <= kelembapan <= 100:
            print("Error: kelembapan harus berada pada rentang 0–100%.")
            return

        # Menyiapkan input dengan nama kolom dan urutan yang sama
        # dengan fitur pada saat training.
        data_input = pd.DataFrame({
            "Suhu (°C)": [suhu],
            "Kelembapan (%)": [kelembapan]
        })

        # Menjalankan inference.
        # Pipeline otomatis melakukan scaling sebelum KNN memprediksi.
        hasil = model_knn.predict(data_input)[0]

        # Menampilkan kembali nilai input agar pengguna dapat
        # memeriksa data yang sedang diproses.
        print("========== HASIL KLASIFIKASI ==========")
        print(f"Suhu input       : {suhu:.2f} °C")
        print(f"Kelembapan input : {kelembapan:.2f} %")
        print(f"Hasil prediksi   : {hasil}")

        # Memberi konteks pada hasil:
        # di luar rentang 20–25°C, label mengikuti aturan awal.
        # Di dalam rentang tersebut, hasil adalah prediksi KNN.
        if suhu < 20:
            print("Keterangan: sesuai label awal Dingin (<20°C).")
        elif suhu > 25:
            print("Keterangan: sesuai label awal Panas (>25°C).")
        else:
            print(
                "Keterangan: suhu berada di rentang 20–25°C. "
                "Hasil merupakan perkiraan KNN, bukan label acuan."
            )

        # Menampilkan probabilitas keluaran model per kelas.
        # Ini adalah proporsi tetangga tiap kelas, bukan jaminan
        # probabilitas kebenaran kondisi sebenarnya.
        probabilitas = model_knn.predict_proba(data_input)[0]
        kelas = model_knn.named_steps["knn"].classes_

        print("\nProporsi suara tetangga per kelas:")
        for nama_kelas, nilai in zip(kelas, probabilitas):
            print(f"{nama_kelas}: {nilai:.2f}")

# Menghubungkan tombol dengan fungsi prediksi.
tombol_proses.on_click(proses_prediksi)

# Menampilkan antarmuka dalam notebook.
display(
    widgets.VBox([
        widgets.HTML("<h3>Klasifikasi Suhu dan Kelembapan dengan KNN</h3>"),
        input_suhu,
        input_kelembapan,
        tombol_proses,
        area_hasil
    ])
)


In [8]:
import sys
import ipywidgets
import IPython

print("Python:", sys.executable)
print("Versi ipywidgets:", ipywidgets.__version__)
print("Versi IPython:", IPython.__version__)

Python: C:\Users\Samsung\.conda\envs\dataku\python.exe
Versi ipywidgets: 8.1.9
Versi IPython: 9.16.1
